# Analisi bayesiana — modello nominale (concordato con l'analisi frequentista)

Modello comune ai due metodi (deciso con Marco, 2026-10-06): finestra $[5100,5600]$ MeV, segnale gaussiano con $\mu,\sigma$ condivisi, fondo con rese indipendenti per carica ($N_b^\pm=N_b(1\mp A_b)/2$) e **doppio esponenziale con forma separata per carica**; likelihood estesa non binnata, prior uniformi. Il modello è in `bayesian_nominal.py`.

Il **modello semplificato** (6 parametri, finestra $[5150,5450]$, un esponenziale, fondo simmetrico) resta in `Bayesian_Analysis.ipynb` / `bayesian_model.py` con tutti i suoi risultati (`results/*.json`), invariato: i suoi 500 toy sono il closure test ad alta statistica della procedura.

Ordine: fit nominale → controlli (Bayes factor, catene disperse, polarità del magnete) → studio sistematico (varianti in parallelo) → toy-MC in parallelo → Dalitz → **passo finale: veto sul charm** (come nel notebook frequentista) → salvataggio. Ogni blocco lungo salva checkpoint riga per riga (`*.jsonl`) e riprende da dove si era fermato.

Modalità di test: `BAYES_NOMINAL_SMOKE=1` usa catene corte e pochi toy e scrive in `results/nominal_smoke/`.

In [ ]:
import os, sys, json, time
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace, asdict

sys.path.insert(0, ".")
import bayesian_nominal as bn

SMOKE = os.environ.get("BAYES_NOMINAL_SMOKE", "0") == "1"
DATA_DIR = "../../opendata-project/Data"
RESULTS_DIR = os.path.join(os.getcwd(), "results", "nominal_smoke" if SMOKE else "nominal")
os.makedirs(RESULTS_DIR, exist_ok=True)

NSTEPS      = 1500 if SMOKE else 25000   # nominal and every variation (chain >= 50 tau for all parameters)
N_TOYS      = 4 if SMOKE else 200
NSTEPS_TOY  = 1500 if SMOKE else 12000   # tau ~ 150-185 at full length: >= 65 tau after burn-in
N_WORKERS   = max(1, (os.cpu_count() or 2) - 1)
N_THINNED   = 2000 if SMOKE else 20000   # posterior sample saved for the figures

def save_json(name, obj):
    path = os.path.join(RESULTS_DIR, name)
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)
    print("[saved]", path)

print("SMOKE =", SMOKE, " RESULTS_DIR =", RESULTS_DIR, " workers =", N_WORKERS)

In [ ]:
cands = bn.load_candidates(DATA_DIR)
data_nom = bn.select(cands, bn.NOMINAL)
print(f"preselected candidates (|q|=1): {len(cands)};  in [{bn.NOMINAL.lo:.0f},{bn.NOMINAL.hi:.0f}] MeV: {len(data_nom)}")
print(f"  B+: {(data_nom.q > 0).sum()}   B-: {(data_nom.q < 0).sum()}   counting asymmetry: "
      f"{((data_nom.q < 0).sum() - (data_nom.q > 0).sum()) / len(data_nom):+.4f}")

## Fit nominale

In [ ]:
# resume: a completed nominal fit with the same configuration and chain length is loaded, not refitted
f_fit, f_th = os.path.join(RESULTS_DIR, "nominal_fit.json"), os.path.join(RESULTS_DIR, "nominal_posterior_thinned.npy")
prev = json.load(open(f_fit)) if os.path.exists(f_fit) and os.path.exists(f_th) else None
same_cfg = prev is not None and prev["config"] == json.loads(json.dumps(asdict(bn.NOMINAL)))  # tuples become lists in JSON
if same_cfg and prev["nsteps"] == NSTEPS:
    nominal = dict(prev, _thinned=np.load(f_th))
    print("[resume] nominal fit loaded from", f_fit)
else:
    nominal = bn.fit(data_nom.m, data_nom.q, bn.NOMINAL, nsteps=NSTEPS, keep_thinned=N_THINNED)
    np.save(f_th, nominal["_thinned"])
    save_json("nominal_fit.json", bn.to_jsonable(nominal))

A = nominal["A"]
print(f"A_raw = {A['median']:.4f}  68% CI [{A['ci68'][0]:.4f}, {A['ci68'][1]:.4f}]  95% CI [{A['ci95'][0]:.4f}, {A['ci95'][1]:.4f}]  sd {A['sd']:.4f}")
print(f"A_b   = {nominal['A_b']['median']:.4f} +- {nominal['A_b']['sd']:.4f}")
print(f"MAP A = {nominal['map']['A']:.4f}   (con prior piatti MAP = MLE: confronto diretto con il fit frequentista)")
print(f"chi2/ndof = {nominal['gof']['chi2']:.1f}/{nominal['gof']['ndof']}  p = {nominal['gof']['p']:.1e}")
print(f"acceptance {nominal['acceptance']:.2f};  chain/tau minimo = {nominal['chain_over_tau_min']:.0f} (deve essere > 50)")
for k, t in nominal["tau"].items():
    print(f"   tau[{k:9s}] = {t:6.1f}")
if not SMOKE:
    assert nominal["chain_over_tau_min"] > 50, "catena troppo corta rispetto a tau: aumentare NSTEPS"

In [ ]:
import corner
th = nominal["_thinned"]
fig = corner.corner(th, labels=nominal["names"], quantiles=[0.16, 0.5, 0.84], show_titles=True,
                    plot_datapoints=False, title_fmt=".4g")
plt.show()

## Bayes factor (Savage-Dickey, approssimazione gaussiana)

$A=0$ e $A_b=0$ contro parametro libero, prior uniforme su $(-1,1)$ (densità $1/2$), odds a priori uguali.

In [ ]:
bf_A = bn.bayes_factor_gaussian(nominal, "A", 1 / bn.prior_volume_A(bn.NOMINAL))
bf_Ab = bn.bayes_factor_gaussian(nominal, "A_b", 0.5)
print(f"A = 0  : BF_01 = {bf_A:.2e}   BF_10 = {1/bf_A:.2e}   ({abs(A['mean'])/A['sd']:.1f} sd da zero)")
print(f"A_b = 0: BF_01 = {bf_Ab:.2f}   BF_10 = {1/bf_Ab:.2f}   ({abs(nominal['A_b']['mean'])/nominal['A_b']['sd']:.1f} sd da zero)")

## Controlli e studio sistematico (in parallelo)

Un unico lotto di fit indipendenti, ciascuno con la stessa lunghezza di catena del nominale:
- due fit con i walker distribuiti uniformemente su tutto il box del prior (Week 4: più catene da punti diversi);
- i due campioni di polarità del magnete (controllo incrociato, non sistematica);
- le varianti dello studio sistematico (`bn.VARIATIONS`);
- il passo finale: veto sul charm (`bn.CHARM_VETO_FINAL`).

In [ ]:
fit_tasks = []
for s in (1, 2):
    fit_tasks.append((replace(bn.NOMINAL, name=f"dispersed_{s}"), data_nom.m.to_numpy(), data_nom.q.to_numpy(), NSTEPS, s, True))
for pol in ("down", "up"):
    d = data_nom[data_nom.magnet == pol]
    fit_tasks.append((replace(bn.NOMINAL, name=f"magnet_{pol}"), d.m.to_numpy(), d.q.to_numpy(), NSTEPS, 42, False,
                      os.path.join(RESULTS_DIR, f"magnet_{pol}_A_thinned.npy")))
for cfg in bn.VARIATIONS + [bn.CHARM_VETO_FINAL]:
    d = bn.select(cands, cfg)
    fit_tasks.append((cfg, d.m.to_numpy(), d.q.to_numpy(), NSTEPS, 42, False))

fits = bn.run_parallel(fit_tasks, bn._fit_task, os.path.join(RESULTS_DIR, "fits.jsonl"),
                       key=lambda r: r["config"]["name"], n_workers=N_WORKERS)
fits = {r["config"]["name"]: r for r in fits}
for name, r in fits.items():
    print(f"{name:22s} n={r['n']:6d}  A={r['A']['median']:+.4f} [{r['A']['ci68'][0]:+.4f},{r['A']['ci68'][1]:+.4f}]  "
          f"dA={r['A']['median']-A['median']:+.4f}  chi2/ndof={r['gof']['chi2']:.0f}/{r['gof']['ndof']}  "
          f"chain/tau={r['chain_over_tau_min']:.0f}" + (f"  bulk dopo {r['steps_to_bulk']} step" if r.get('dispersed') else ""))

In [ ]:
syst = bn.systematics(nominal, [fits[c.name] for c in bn.VARIATIONS])
print("spostamenti:", {k: round(v, 4) for k, v in syst["shifts"].items()})
print("per gruppo :", {k: round(v, 4) for k, v in syst["groups_max_abs_shift"].items()})
print(f"sistematica totale ({syst['rule']}): {syst['total_quadrature']:.4f}   vs statistica {A['sd']:.4f}")

## Toy-MC (in parallelo)

Pseudo-dataset generati dal modello nominale con i parametri alla mediana a posteriori; ogni toy è rifittato con lo stesso modello. Coverage degli intervalli al 68%/95%, bias e dispersione delle stime. È un closure test: non verifica l'adeguatezza del modello ai dati.

In [ ]:
theta_true = {k: nominal[k]["median"] for k in nominal["names"]}
toy_tasks = [(i, theta_true, bn.NOMINAL, NSTEPS_TOY, 1000 + i) for i in range(N_TOYS)]
toys = bn.run_parallel(toy_tasks, bn._toy_task, os.path.join(RESULTS_DIR, "toys.jsonl"),
                       key=lambda r: r["toy"], n_workers=N_WORKERS, log_every=10)
cov = bn.coverage_summary(toys, theta_true["A"], A["sd"])
print(json.dumps(cov, indent=1))

## Asimmetria locale nel piano di Dalitz (binning fisso $8\times8$)

Stessa procedura del modello semplificato, sulla finestra nominale: significatività di ogni bin rispetto all'asimmetria globale e test di omogeneità; ripetuto nella finestra di segnale ($|m-\mu|<2\sigma$, con $\mu,\sigma$ del fit nominale) e con i veti sul charm.

In [ ]:
from scipy.stats import chi2 as _chi2, binom
def dalitz_test(d, label, min_events=20):
    lo2, hi2 = (np.minimum(d.m12, d.m13) / 1e3) ** 2, (np.maximum(d.m12, d.m13) / 1e3) ** 2
    q = d.q.to_numpy(); ag = (np.sum(q < 0) - np.sum(q > 0)) / len(q)
    el, eh = np.linspace(0, 16, 9), np.linspace(0, 30, 9)
    npl, _, _ = np.histogram2d(lo2[q > 0], hi2[q > 0], [el, eh]); nmi, _, _ = np.histogram2d(lo2[q < 0], hi2[q < 0], [el, eh])
    n = npl + nmi; k = n >= min_events
    ai = (nmi - npl)[k] / n[k]; s = (ai - ag) / np.sqrt((1 - ai ** 2) / n[k])
    c2 = float(np.sum(s ** 2)); nd = int(k.sum() - 1); nsig = int(np.sum(np.abs(s) > 3))
    out = {"label": label, "n": int(len(q)), "A_glob": float(ag), "bins_populated": int(k.sum()), "bins_3sigma": nsig,
           "chi2": c2, "ndof": nd, "p": float(_chi2.sf(c2, nd)),
           "p_at_least_nsig_if_uniform": float(binom.sf(nsig - 1, int(k.sum()), 2 * 0.00135)) if nsig else 1.0}
    print(f"{label:34s} n={out['n']:6d} A_glob={ag:+.4f}  {nsig}/{k.sum()} bin |S|>3  chi2={c2:.1f}/{nd}  p={out['p']:.1e}")
    return out

mu_n, sg_n = nominal["mu"]["median"], nominal["sigma"]["median"]
dalitz = [dalitz_test(data_nom, "finestra nominale"),
          dalitz_test(data_nom[np.abs(data_nom.m - mu_n) < 2 * sg_n], "finestra di segnale"),
          dalitz_test(bn.select(cands, replace(bn.NOMINAL, veto=("D0",))), "veto D0"),
          dalitz_test(bn.select(cands, replace(bn.NOMINAL, veto=("D0", "Jpsi", "chic0"))), "veto D0, J/psi, chi_c0")]

## Passo finale: veto sul charm

Risultato charmless: fit nominale sul campione senza i candidati con una coppia $K^+K^-$ di carica opposta entro `veto_halfwidth` dalla massa degli stati in `bn.CHARM_VETO_FINAL.veto` (da allineare con il notebook frequentista). Il fit è nel lotto parallelo sopra.

In [ ]:
fv = fits[bn.CHARM_VETO_FINAL.name]
print(f"veto {bn.CHARM_VETO_FINAL.veto} ±{bn.CHARM_VETO_FINAL.veto_halfwidth:.0f} MeV: n={fv['n']}  "
      f"A_raw = {fv['A']['median']:.4f}  68% CI [{fv['A']['ci68'][0]:.4f}, {fv['A']['ci68'][1]:.4f}]  "
      f"(spostamento {fv['A']['median'] - A['median']:+.4f} rispetto al nominale senza veto)")

## Salvataggio

In [ ]:
summary = {
    "model": asdict(bn.NOMINAL), "smoke": SMOKE, "nsteps": NSTEPS, "nsteps_toy": NSTEPS_TOY,
    "n_candidates_window": len(data_nom),
    "nominal": bn.to_jsonable(nominal),
    "bayes_factor": {"A0_BF01": bf_A, "Ab0_BF01": bf_Ab},
    "checks": {k: fits[k] for k in ("dispersed_1", "dispersed_2", "magnet_down", "magnet_up")},
    "variations": {c.name: fits[c.name] for c in bn.VARIATIONS},
    "systematics": syst,
    "toys": cov,
    "dalitz": dalitz,
    "charm_veto_final": fits[bn.CHARM_VETO_FINAL.name],
}
expected = ["nominal", "bayes_factor", "checks", "variations", "systematics", "toys", "dalitz", "charm_veto_final"]
assert all(summary.get(k) is not None for k in expected), "manca un risultato nel riepilogo"
save_json("nominal_summary.json", summary)